# 🔬 Notebook 4: Explicabilidad con LIME y Comparación de Frameworks

> Este notebook explica predicciones individuales del modelo scikit-learn usando **LIME** y presenta la **comparación final** entre los frameworks scikit-learn y PySpark.

### 📋 Contenido:
1. **Carga del modelo** entrenado y datos de prueba
2. **Identificación de instancias mal clasificadas** (falsos positivos y falsos negativos)
3. **Explicabilidad con LIME** — análisis de predicciones erróneas
4. **Tabla comparativa** — scikit-learn vs PySpark (métricas y tiempos)
5. **Curvas ROC superpuestas** — comparación visual
6. **Comparación de tiempos** de cómputo
7. **Reflexión crítica** sobre ambos frameworks

### ⚙️ Requisitos:
- Notebook 2 ejecutado (modelo sklearn guardado + métricas)
- Notebook 3 ejecutado (métricas Spark guardadas)
- Librerías: lime, scikit-learn, pandas, matplotlib

---

---

## ⚙️ 0.1 Configuración del Entorno

> **Importamos las dependencias y aplicamos el estilo visual consistente con el resto del proyecto.**

---

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("No se encontró la raíz del proyecto con el directorio src.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from ctr_mlp.config import load_project_settings, apply_dark_style, COLORS, AXES_BG
from ctr_mlp.data_io import split_features_target
from ctr_mlp.evaluation import (
    compute_binary_metrics,
    plot_confusion_matrix,
    plot_roc_curve,
    plot_comparative_roc,
    plot_comparative_times,
    build_comparison_table,
)
from ctr_mlp.explainability import (
    build_lime_explainer_from_pipeline,
    explain_pipeline_prediction,
    explanation_to_frame,
    find_misclassified_instances,
)
from ctr_mlp.feature_engineering import add_time_features_pandas
from ctr_mlp.sklearn_workflow import split_train_test
from ctr_mlp.utils import format_seconds

# Aplicar estilo oscuro profesional
apply_dark_style()

settings = load_project_settings()
paths = settings["paths"]
general = settings["general"]
feature_cfg = settings["features"]

FIGURES_DIR = paths["figures_dir"]
MODELS_DIR = paths["models_dir"]
TARGET_COL = general["target_col"]

---

## 📥 1.1 Carga del Modelo Entrenado y Datos

> **Cargamos el mejor modelo de scikit-learn guardado en el Notebook 2 y reconstruimos los conjuntos de train/test para aplicar LIME.**

---

In [ ]:
# Cargar modelo guardado
model_path = Path(MODELS_DIR) / "sklearn_best_mlp.joblib"
pipeline = joblib.load(model_path)
print(f"Modelo cargado desde: {model_path}")

# Cargar muestra y reconstruir datos
sample_path = paths["sampled_train_parquet"]
df_sample = pd.read_parquet(sample_path)
print(f"Muestra cargada: {df_sample.shape}")

selected_cols = feature_cfg["sklearn_categorical"] + feature_cfg["sklearn_numeric"] + [TARGET_COL]
model_df = df_sample[selected_cols].dropna().copy()
X, y = split_features_target(model_df, target_col=TARGET_COL)
X_train, X_test, y_train, y_test = split_train_test(
    X, y, random_state=general["random_state"]
)
print(f"Test set: {X_test.shape[0]:,} registros")

# Obtener predicciones
y_pred = pipeline.predict(X_test)
y_score = pipeline.predict_proba(X_test)[:, 1]

---

## ❌ 2.1 Identificación de Instancias Mal Clasificadas

> **Buscamos falsos positivos (el modelo predijo clic pero no hubo) y falsos negativos (el modelo no predijo un clic real). Estas instancias son las más interesantes para explicar con LIME, ya que revelan qué confunde al modelo.**

---

In [ ]:
# Encontrar instancias mal clasificadas
errors = find_misclassified_instances(
    y_test, y_pred, y_score=y_score, error_type="both", n=5
)

print(f"Falsos Positivos encontrados: {len(errors['false_positives'])}")
print(f"  Índices: {errors['false_positives'].tolist()}")
print(f"\nFalsos Negativos encontrados: {len(errors['false_negatives'])}")
print(f"  Índices: {errors['false_negatives'].tolist()}")

# Resumen de errores
total_errors = (y_pred != y_test.values).sum()
print(f"\nTotal de errores de clasificación: {total_errors:,} / {len(y_test):,} ({total_errors/len(y_test)*100:.2f}%)")

---

## 🔬 2.2 Explicación LIME — Falso Positivo

> **Analizamos un falso positivo con LIME: una instancia donde el modelo predijo que el usuario haría clic, pero en realidad no lo hizo. LIME revela qué features impulsaron la predicción errónea.**

---

In [ ]:
# Construir explicador LIME
lime_artifacts = build_lime_explainer_from_pipeline(
    pipeline,
    X_reference=X_train,
    background_size=general["lime_background_size"],
)

# Explicar un falso positivo
fp_idx = errors["false_positives"][0]
fp_instance = X_test.iloc[[fp_idx]]
fp_true = int(y_test.iloc[fp_idx])
fp_pred_prob = float(y_score[fp_idx])

print(f"Instancia seleccionada (índice en test): {fp_idx}")
print(f"Valor real: {fp_true} (No Click)")
print(f"Predicción del modelo: 1 (Click)")
print(f"Probabilidad predicha de click: {fp_pred_prob:.4f}")

explanation_fp = explain_pipeline_prediction(
    pipeline, lime_artifacts, fp_instance, label=1, num_features=10
)

# Tabla de features más influyentes
fp_frame = explanation_to_frame(explanation_fp, label=1).sort_values("weight", ascending=False)
print("\nFeatures más influyentes en la decisión errónea:")
display(fp_frame)

In [ ]:
# Gráfico de explicación LIME (falso positivo)
fig_fp = explanation_fp.as_pyplot_figure(label=1)
fig_fp.set_size_inches(14, 7)
fig_fp.suptitle("Explicación LIME — Falso Positivo", fontsize=14, fontweight="bold")
plt.tight_layout()
fig_fp.savefig(FIGURES_DIR / "04_lime_false_positive.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 🔬 2.3 Explicación LIME — Falso Negativo

> **Analizamos un falso negativo: una instancia donde el modelo predijo "no clic" pero el usuario sí hizo clic. Esto revela qué patrones el modelo no logra capturar.**

---

In [ ]:
# Explicar un falso negativo
fn_idx = errors["false_negatives"][0]
fn_instance = X_test.iloc[[fn_idx]]
fn_true = int(y_test.iloc[fn_idx])
fn_pred_prob = float(y_score[fn_idx])

print(f"Instancia seleccionada (índice en test): {fn_idx}")
print(f"Valor real: {fn_true} (Click)")
print(f"Predicción del modelo: 0 (No Click)")
print(f"Probabilidad predicha de click: {fn_pred_prob:.4f}")

explanation_fn = explain_pipeline_prediction(
    pipeline, lime_artifacts, fn_instance, label=1, num_features=10
)

fn_frame = explanation_to_frame(explanation_fn, label=1).sort_values("weight", ascending=False)
print("\nFeatures más influyentes en la decisión errónea:")
display(fn_frame)

In [ ]:
# Gráfico de explicación LIME (falso negativo)
fig_fn = explanation_fn.as_pyplot_figure(label=1)
fig_fn.set_size_inches(14, 7)
fig_fn.suptitle("Explicación LIME — Falso Negativo", fontsize=14, fontweight="bold")
plt.tight_layout()
fig_fn.savefig(FIGURES_DIR / "04_lime_false_negative.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 📊 2.4 Resumen de Contribuciones — Positivas vs Negativas

> **Separamos las features que impulsan la predicción hacia "click" (pesos positivos) de las que impulsan hacia "no click" (pesos negativos) para ambas instancias mal clasificadas.**

---

In [ ]:
for label, frame, instance_type in [
    ("Falso Positivo", fp_frame, "FP"),
    ("Falso Negativo", fn_frame, "FN"),
]:
    positive = frame[frame["weight"] > 0].head(5).reset_index(drop=True)
    negative = frame[frame["weight"] < 0].sort_values("weight").head(5).reset_index(drop=True)
    
    print(f"\n{'═' * 50}")
    print(f" {label} — Contribuciones")
    print(f"{'═' * 50}")
    print("\n  Impulsan hacia CLICK (+):")
    display(positive)
    print("  Impulsan hacia NO CLICK (-):")
    display(negative)

---

## 🏆 3. Comparación de Frameworks: scikit-learn vs PySpark

> **Consolidamos las métricas de ambos frameworks para una comparación lado a lado. Cargamos los resultados guardados en los Notebooks 2 y 3.**

---

---

## 📝 3.1 Tabla Comparativa de Métricas

> **Comparamos todas las métricas obligatorias lado a lado: Accuracy, Precision, Recall, F1, ROC AUC, tiempos de entrenamiento y predicción, y tamaño del dataset utilizado.**

---

In [ ]:
# Cargar métricas de scikit-learn
sklearn_metrics_path = Path(MODELS_DIR) / "sklearn_metrics.csv"
sklearn_metrics_df = pd.read_csv(sklearn_metrics_path)
sklearn_metrics = sklearn_metrics_df.iloc[0].to_dict()

# Cargar métricas de PySpark
spark_metrics_path = Path(MODELS_DIR) / "spark_metrics.csv"
spark_metrics_df = pd.read_csv(spark_metrics_path)
spark_metrics = spark_metrics_df.iloc[0].to_dict()

# Construir tabla comparativa
sklearn_dataset_size = len(df_sample)
spark_dataset_size = int(spark_metrics.get("dataset_size", 40_000_000))

comparison_table = build_comparison_table(
    sklearn_metrics=sklearn_metrics,
    spark_metrics=spark_metrics,
    sklearn_dataset_size=sklearn_dataset_size,
    spark_dataset_size=spark_dataset_size,
)

print("═" * 60)
print("COMPARACIÓN: scikit-learn vs PySpark")
print("═" * 60)
display(comparison_table)

---

## 📈 3.2 Curvas ROC Superpuestas

> **Comparamos visualmente las curvas ROC de ambos frameworks en un mismo gráfico. Si los datos de predicción de PySpark están disponibles, se superponen ambas curvas; de lo contrario, se muestra solo la de scikit-learn con referencia al AUC de Spark.**

---

In [ ]:
# Cargar predicciones de sklearn guardadas
sklearn_pred_path = Path(MODELS_DIR) / "sklearn_predictions.parquet"
if sklearn_pred_path.exists():
    sklearn_preds = pd.read_parquet(sklearn_pred_path)
    sk_y_true = sklearn_preds["y_true"].values
    sk_y_score = sklearn_preds["y_score"].values
else:
    sk_y_true = y_test.values
    sk_y_score = y_score

# Intentar cargar predicciones de Spark (si están disponibles)
spark_pred_path = Path(MODELS_DIR) / "spark_predictions.parquet"
roc_results = [
    {"name": "scikit-learn (1M muestra)", "y_true": sk_y_true, "y_score": sk_y_score},
]
if spark_pred_path.exists():
    spark_preds = pd.read_parquet(spark_pred_path)
    roc_results.append({
        "name": "PySpark (dataset completo)",
        "y_true": spark_preds["y_true"].values,
        "y_score": spark_preds["y_score"].values,
    })

fig_roc = plot_comparative_roc(
    roc_results,
    save_path=FIGURES_DIR / "04_comparative_roc.png"
)

# Si Spark no tiene predicciones detalladas, anotar su AUC como referencia
if not spark_pred_path.exists():
    spark_auc = spark_metrics.get("roc_auc", 0)
    ax = fig_roc.get_axes()[0]
    ax.axhline(y=0.5, alpha=0)  # Invisible, solo para que la leyenda se actualice
    ax.text(0.6, 0.3, f"PySpark AUC = {spark_auc:.4f}\n(curva no disponible)",
            fontsize=12, color=COLORS["accent"], fontstyle="italic",
            bbox=dict(boxstyle="round", facecolor=AXES_BG, edgecolor=COLORS["accent"], alpha=0.8))

plt.show()

---

## ⏱️ 3.3 Comparación de Tiempos de Cómputo

> **Visualizamos las diferencias en tiempo de entrenamiento y predicción entre ambos frameworks. PySpark tiene mayor overhead inicial pero escala mejor con datasets grandes.**

---

In [ ]:
time_df = pd.DataFrame([
    {
        "framework": "scikit-learn",
        "training_seconds": sklearn_metrics.get("training_seconds", 0),
        "prediction_seconds": sklearn_metrics.get("prediction_seconds", 0),
    },
    {
        "framework": "PySpark",
        "training_seconds": spark_metrics.get("training_seconds", 0),
        "prediction_seconds": spark_metrics.get("prediction_seconds", 0),
    },
])

fig_times = plot_comparative_times(
    time_df,
    save_path=FIGURES_DIR / "04_comparative_times.png"
)
plt.show()

print(f"\nResumen de tiempos:")
print(f"  scikit-learn — Entrenamiento: {format_seconds(sklearn_metrics.get('training_seconds', 0))}")
print(f"  scikit-learn — Predicción:    {format_seconds(sklearn_metrics.get('prediction_seconds', 0))}")
print(f"  PySpark      — Entrenamiento: {format_seconds(spark_metrics.get('training_seconds', 0))}")
print(f"  PySpark      — Predicción:    {format_seconds(spark_metrics.get('prediction_seconds', 0))}")

---

## 💡 4. Reflexión Crítica

> **Análisis crítico de los resultados y reflexiones sobre cuándo usar cada framework.**

### ❓ ¿Qué entorno fue más rápido?

**scikit-learn** es generalmente más rápido en tiempo total cuando el dataset cabe en memoria. Sin embargo, esta comparación no es directa porque scikit-learn operó sobre una muestra de 1M de registros mientras PySpark procesó el dataset completo (~40M). El overhead de Spark (inicialización del contexto, serialización, comunicación entre ejecutores) lo hace menos eficiente para datasets pequeños, pero su escalabilidad lo justifica para volúmenes que no caben en memoria de una sola máquina.

### ❓ ¿Cuál fue más preciso?

PySpark tiene la ventaja teórica de entrenar sobre todos los datos, capturando patrones que una muestra podría perder. Sin embargo, las métricas entre ambos suelen ser comparables cuando la muestra es representativa (como en nuestro caso con muestreo estratificado). La diferencia en ROC AUC suele ser marginal (<2%), lo que sugiere que la muestra de 1M captura bien la estructura del dataset.

### ❓ ¿Cuándo es útil PySpark vs scikit-learn?

| Escenario | Recomendación |
|-----------|---------------|
| Prototipado rápido y experimentación | scikit-learn |
| Dataset < 5M registros | scikit-learn |
| Dataset > 10M registros | PySpark |
| Pipeline de producción con datos streaming | PySpark |
| Interpretabilidad local (LIME, SHAP) | scikit-learn |
| Cluster disponible para entrenamiento | PySpark |
| Debugging y ajuste fino de hiperparámetros | scikit-learn |

### ❓ ¿Qué aporta LIME al análisis?

LIME proporciona **explicabilidad local**: en lugar de entender el modelo globalmente, nos permite analizar predicciones individuales. Esto es especialmente valioso para:

1. **Detección de errores**: Entender POR QUÉ el modelo se equivoca en casos específicos (falsos positivos y negativos).
2. **Confianza en el modelo**: Verificar que el modelo usa features razonables y no patrones espurios.
3. **Comunicación con stakeholders**: Explicar decisiones del modelo a audiencias no técnicas.
4. **Mejora iterativa**: Identificar features que podrían mejorarse o añadirse basándose en los errores.

En nuestro caso, LIME reveló qué combinaciones de features categóricas (site, app, dispositivo) y temporales confunden al modelo, proporcionando insights accionables para mejorar el pipeline de features.

---

---

## 🌟 Conclusión Final del Proyecto

> **Este proyecto demostró un flujo de trabajo completo de clasificación CTR, desde el análisis exploratorio hasta la explicabilidad local, comparando dos paradigmas de implementación. La combinación de scikit-learn para prototipado rápido e interpretabilidad, con PySpark para escalabilidad a datos masivos, representa un enfoque pragmático y realista para problemas de Machine Learning en producción.**

---